# 06 · Organocatalysis: transferring a TS across scaffolds

The capability the whole design is built for: take a known catalytic geometry and transfer its conserved reacting core onto a different catalyst backbone. Here the isothiourea family (tetramisole → BTM → HyperBTM); Lewis-base acyl-transfer catalysts sharing an amidine–thioether core that activates substrates through an S···C=O chalcogen bond. Swap the backbone, conserve the core. Renders highlight the conserved core; the S···O contact is an NCI (`nci_bonds`, dotted); a real TS's reacting bonds use `ts_bonds` (dashed).

In [ ]:
import tempfile
import numpy as np
import rxembed.pipeline as rx
import xyzrender
from rdkit import Chem, RDLogger
from rdkit.Chem import rdMolTransforms as T
from rxembed.pipeline.perceive import _xyz_to_mol

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")
CATALYSTS = {  # the isothiourea family; all share the amidine-thioether core, distinct backbones
    "tetramisole": "C1CSC2=NC(CN12)c1ccccc1",  # imidazo[2,1-b]thiazoline (no benzo fusion)
    "BTM": "C1CN2C(=NC1c1ccccc1)Sc1ccccc12",  # benzotetramisole: the thiazole is benzo-fused
    "HyperBTM": "CC(C)[C@]1CN2C(=N[C@]1c1ccccc1)Sc1ccccc12",  # BTM + an i-Pr stereocentre
}
CORE = "[#7][#6]([#16])=[#7]"  # the conserved amidine-thioether reacting core (N-C(S)=N)


def core_idx(mol):
    return [i + 1 for i in mol.GetSubstructMatch(Chem.MolFromSmarts(CORE))]  # 1-based, for highlight

## The catalyst family

Each backbone embeds and conformer-searches like anything else; the highlighted amidine–thioether core does the chemistry and is conserved across all of them.

In [ ]:
for name, smi in CATALYSTS.items():
    ens = rx.embed(smi, stereo="free").mc(preset="rapid").prune()  # free: the undefined ring
    print(f"  {name:12s}: {len(ens):2d} conformers ({Chem.MolFromSmiles(smi).GetNumAtoms()} heavy atoms)")
hb = rx.embed(CATALYSTS["HyperBTM"]).mc(preset="rapid").prune()  # stereocentre is @-specified here
xyzrender.render(hb.lowest(1).dump(tempfile.mktemp(suffix=".xyz")), no_hy=True, highlight=core_idx(hb.mol))

## The activation: an S···C=O chalcogen bond

Isothiourea catalysis runs through a chalcogen-bonded acyl-ammonium: the ring S σ-hole grips the substrate carbonyl O. Add an acyl donor and let `nci_candidates` find that directional contact.

In [ ]:
catalyst = CATALYSTS["tetramisole"]
acyl_donor = "CC(=O)OC(C)=O"  # acetic anhydride: the acyl source
complex_smi = f"{catalyst}.{acyl_donor}"
cands = rx.nci_candidates(Chem.AddHs(Chem.MolFromSmiles(complex_smi)))
chb = next(k for k in cands if k.startswith("ChB"))  # the S...O=C chalcogen bond
print(f"activation contact: {chb}  {cands[chb].distances}")
# stereo="free": tetramisole's ring stereocentre is undefined in the SMILES and irrelevant to the
# chalcogen grip; without it rx.embed returns an EnsembleSet of both enantiomers, not one Ensemble.
ens = rx.embed(complex_smi, contacts=cands[chb], n=8, stereo="free").mc(preset="rapid").prune()
s, o = next(iter(cands[chb].distances))
print(f"{len(ens)} conformers; S···O = {ens.measure((s, o))['mean']:.2f} Å")
xyzrender.render(
    ens.lowest(1).dump(tempfile.mktemp(suffix=".xyz")),
    no_hy=True,
    nci_bonds=[(s + 1, o + 1)],
    labels=[f"{s + 1} {o + 1} d"],
)  # the chalcogen contact (dotted)

## The backbone swap: the headline

Embed one scaffold (tetramisole), then transfer its amidine-core geometry onto the other backbones; `template=` the geometry, an index map the conserved core. The core is pinned exactly (dev ≈ 0, highlighted) while each new backbone's periphery is searched: one known geometry → a whole catalyst series.

In [ ]:
template = rx.embed(CATALYSTS["tetramisole"], stereo="free").mc(preset="rapid").prune()
q = Chem.MolFromSmarts(CORE)
tm = template.mol.GetSubstructMatch(q)  # the conserved N-C(S)=N core in the template geometry


def core_dev(ens):
    nm = ens.mol.GetSubstructMatch(q)
    tp = template.mol.GetConformer(template.ids[0]).GetPositions()
    pw = lambda p_, ix: sorted(np.linalg.norm(p_[a] - p_[b]) for i, a in enumerate(ix) for b in ix[i + 1 :])
    return max(
        max(abs(x - y) for x, y in zip(pw(ens.mol.GetConformer(c).GetPositions(), nm), pw(tp, tm))) for c in ens.ids
    )


swaps = {}
for name in ["BTM", "HyperBTM"]:
    nm = Chem.MolFromSmiles(CATALYSTS[name]).GetSubstructMatch(q)  # the same core, matched on each backbone
    sw = rx.embed(CATALYSTS[name], template=(template, dict(zip(nm, tm))), stereo="free").mc(preset="rapid").prune()
    swaps[name] = sw
    print(f"  tetramisole -> {name:9s}: {len(sw):2d} confs | conserved amidine core dev {core_dev(sw):.4f} \u00c5")
sw = swaps["HyperBTM"]
xyzrender.render(sw.lowest(1).dump(tempfile.mktemp(suffix=".xyz")), no_hy=True, highlight=core_idx(sw.mol))

## Rank the transferred poses: real energy closes the loop

The core is pinned and each periphery searched, but which transferred pose wins? Score with GFN-FF (NCI-aware) to rank, then a g-xTB single point on the top few. Bias the seed with the template, let a real energy decide, the whole design principle, end to end.

In [ ]:
ranked = swaps["HyperBTM"].score("gfnff")  # rank the templated poses on a real, NCI-aware energy
print(
    f"tetramisole->HyperBTM: {ranked.n} poses | GFN-FF ΔE {sorted(round(e, 1) for e in ranked.relative().values())[:5]} kcal/mol"
)
best = ranked.lowest(2).score("gxtb")  # g-xTB single point on the top 2
print(f"g-xTB on the top 2: ΔE {sorted(round(e, 2) for e in best.relative().values())} kcal/mol")

## The same pattern for a real DFT TS

The swap above starts from an embedded scaffold; with a genuine TS `.xyz` it is the identical call with `fix=`. A chiral-phosphoric-acid organocatalysis TS (103 atoms) embeds with its reacting core held to 0.000 Å while the catalyst periphery is searched: the reacting bonds drawn with `ts_bonds`.

In [ ]:
cpa = "structures/cpa.xyz"
rc = [23, 35, 70, 74, 75]  # reacting-core atoms (from a graphrc trajectory in practice)
bonds = [(23, 35), (35, 75), (70, 74)]  # forming / breaking bonds held during embed
ens = rx.embed(cpa, fix=rc, n=2).mc(preset="rapid").prune()
pin = _xyz_to_mol(cpa, 0).GetConformer().GetPositions()
drift = max(
    abs(T.GetBondLength(ens.mol.GetConformer(c), i, j) - float(np.linalg.norm(pin[i] - pin[j])))
    for c in ens.ids
    for i, j in bonds
)
print(f"CPA organocatalysis TS (103 atoms): {len(ens)} conformers | reacting core held to {drift:.4f} Å")
clean = [c for c in ens.ids if rx.geom_check.check(ens.mol, conf_id=c, frozen=rc, reference=cpa).ok()]
print(  # the recurring flag is ONE sp2 (C73) next to the held core, ~0.16 Å out of plane vs a 0.15 Å
    f"geometry.check (free periphery): {len(clean)}/{len(ens.ids)} clean"  # limit: the graft's own
    ": the rest pyramidalise the sp2 adjacent to the frozen core"  # strain, caught, not hidden
)
xyzrender.render(
    ens.lowest(1).dump(tempfile.mktemp(suffix=".xyz")), no_hy=True, ts_bonds=[(i + 1, j + 1) for i, j in bonds]
)